# **YOLO 파인튜닝**

# 1.환경준비

## (1) 라이브러리 설치

In [ ]:
!pip install ultralytics roboflow -q

## (2) 라이브러리 불러오기

In [ ]:
from ultralytics import settings, YOLO
from roboflow import Roboflow
import matplotlib.pyplot as plt
import cv2
import os
from IPython.display import Video

* 폴더 내 이미지 개수 확인

In [ ]:
def image_count(path) :
    image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]  # YOLO에서 지원하는 이미지 확장자

    # valid 폴더에서 이미지 파일 개수 확인
    image_count = len([f for f in os.listdir(path) if f.lower().endswith(tuple(image_extensions))])

    return image_count

## (3) YOLO 설정

* 파일 경로 설정

In [ ]:
# 현재 세팅을 확인해 봅시다.
settings

In [ ]:
# 콜랩 파일 탭에 보이는 경로('/content/')로 변경해 봅시다.
settings['datasets_dir'] = '/content/'
settings.update()
settings

# 2.모델링

## (1) 데이터셋 다운로드

![](https://i.imgur.com/xIudTbe.png)

* 아래 링크를 눌러 데이터셋 다운로드 코드를 받아 옵시다.

    - **[링크](https://universe.roboflow.com/roboflow-58fyf/rock-paper-scissors-sxsw)**

In [ ]:
rf = Roboflow(api_key="your key") # 개인별 api key값을 넣습니다.
project = rf.workspace("roboflow-58fyf").project("rock-paper-scissors-sxsw")
version = project.version(14)
dataset = version.download("yolov11")

* 이미지 개수 확인

In [ ]:
# train 이미지
cnt = image_count('/content/rock-paper-scissors-14/train/images')
print('* train 이미지 수 :', cnt)

# valid 이미지
cnt = image_count('/content/rock-paper-scissors-14/valid/images')
print('* valid 이미지 수 :', cnt)

# test 이미지
cnt = image_count('/content/rock-paper-scissors-14/test/images')
print('* test 이미지 수 :', cnt)

## (2) 모델

### 1) 모델 다운로드

- 모델의 구조와 해당 구조에 맞게 사전 학습된 가중치를 불러온다.
- Parameters
    * model : 모델 구조 또는 모델 구조 + 가중치 설정. task와 맞는 모델을 선택해야 한다.
    * task : detect, segment, classify, pose 중 택일

In [ ]:
model = YOLO(model='yolo11n.pt', task='detect')

### 2) 모델 살펴보기

**다음 내용은 참조하세요.**
* Backbone : CNN 기반 특징 추출기 (0~9)
    * 입력 이미지를 받아서 엣지, 텍스처, 형태 같은 저수준~중간 수준의 특징을 뽑음.
    * 보통 Conv, C3 블록, SPPF 같은 모듈이 포함됨.
    * YOLO 에서 0~9번 레이어 (Conv → C3k2 → … → SPPF) 가 여기에 해당
* Neck : 다중 스케일 특징 결합기 (10~22)
    * 여러 단계의 feature map을 서로 결합해서 더 풍부한 표현을 만듦.
    * FPN이나 PAN 구조를 써서 작은 물체부터 큰 물체까지 다 잡을 수 있도록 함.
    * Upsample, Concat, C3k2 같은 블록들이 여기에 해당 (10~22번).
* Head : 최종 예측기 (23)
    * 실제로 bounding box 좌표와 클래스 확률을 출력하는 부분.
    * YOLO에서는 Detect 모듈 (23번)이 여기에 해당.

* 요약 정보 (레이어 수, 파라미터 수 등)

In [ ]:
model.info()

* 모델 블록 단위 확인하기
    * 블록 안에 하위 레이어들이 포함된다.

In [ ]:
# 레이어 구조 확인
for i, m in enumerate(model.model.model):
    print(f"{i:3d} : {m.__class__.__name__}")

* 전체 네트워크 아키텍처 출력

In [ ]:
print(model.model)

## (3) 파인튜닝

* 모델 학습
    * 파라미터 설명 : [Parameters](https://docs.ultralytics.com/modes/train/#train-settings)

In [ ]:
results_train = model.train(model='/content/yolov11n.pt',
                            data='/content/rock-paper-scissors-14/data.yaml',
                            epochs=3,
                            seed=20,
                            optimizer = 'Adam',
                            pretrained=True,
                            lr0 = 0.0001
                            )

In [ ]:
valid_folder = "/content/rock-paper-scissors-14/valid/images"  # valid 폴더 경로 지정
image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]  # YOLO에서 지원하는 이미지 확장자

# valid 폴더에서 이미지 파일 개수 확인
image_count = len([f for f in os.listdir(valid_folder) if f.lower().endswith(tuple(image_extensions))])

print(f"Valid 폴더의 이미지 개수: {image_count}")

## (4) 평가

In [ ]:
# 테스트 이미지로 모델 평가하기
results = model.val(data='/content/rock-paper-scissors-14/data.yaml', split="test")

## (5) 예측해보기

In [ ]:
image_path = 'https://cdn.crowdpic.net/detail-thumb/thumb_d_7CBEBAEF3FFE8DD37E4912332908E40F.png'

In [ ]:
result = model.predict(source=image_path, save=True)
result[0].show()  # 탐지된 객체 출력

## (6)[추가] YOLO모델 구조에 처음부터 학습시키기
* 권장하지 않습니다.

* 모델 구조 선택하기

In [ ]:
model_scratch = YOLO(model='yolo11n.pt', task='detect')

* 모델 학습하기

In [ ]:
results_train = model_scratch.train(model='/content/yolo11n.pt',
                                    data='/content/rock-paper-scissors-14/data.yaml',
                                    epochs=1,
                                    patience=5,
                                    optimizer = 'Adam',
                                    seed=20,
                                    pretrained=False,
                                    )

* 예측해보기

In [ ]:
image_path = 'https://cdn.crowdpic.net/detail-thumb/thumb_d_7CBEBAEF3FFE8DD37E4912332908E40F.png'

In [ ]:
results_pred = model_scratch.predict(source=image_path, stream=False, save=True)

In [ ]:
results_pred

# 3.실습

## (1) 데이터셋 다운로드

* [roboflow universe](https://universe.roboflow.com/)에서 원하는 데이터를 하나 선정해서 직접 다운로드 받고 파인튜닝 해 봅시다.
    * 만약 딱히 떠오르는게 없다면 ▶ [football player](https://universe.roboflow.com/ilyes-talbi-ptwsp/futbol-players/dataset/9/download/yolov11)


* 데이터셋 다운로드 코드

## (2) 모델 다운로드

- 모델의 구조와 해당 구조에 맞게 사전 학습된 가중치를 불러온다.
- Parameters
    * model : 모델 구조 또는 모델 구조 + 가중치 설정. task와 맞는 모델을 선택해야 한다.
    * task : detect, segment, classify, pose 중 택일

## (3) 파인튜닝1
* 앞의 모델 코드를 그대로 이용하되, 에포크(epochs)와 학습율(lr0)만 조정한다.

### 1) 학습

* 모델 학습

### 2) 평가

### 3) 예측해보기

* 모델의 성능이 낮은 이유는?
    * 데이터 부족 → 훈련 데이터의 양/질 문제
    * 에포크 → 학습 반복 횟수 부족 문제

## (4) 파인튜닝2
* freeze를 조정해서 학습할 모델의 범위를 잡아 본다.

### 1) 학습

* freeze 조정 : model.train( ... , freeze = 10)
    * COCO와 유사·소량 데이터: freeze=10 권장
    * 도메인이 제법 다르다면 : freeze=0 ~ 5
    * 더 보수적으로 접근 : freeze=16 정도

### 2) 평가

### 3) 예측해보기